## Importing libraries:

In [1]:
import numpy as np
import pandas as pd

## OneHotEncoding:

In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# 1. Indoor Object Detection Dataset Setup
data = {
    'Room_Type': ['Kitchen', 'Bedroom', 'Living Room', 'Kitchen', 'Bedroom'], # Nominal
    'Object_Color': ['Red', 'Blue', 'Green', 'Red', 'Blue'],                  # Nominal
    'Object_Size': ['Small', 'Large', 'Medium', 'Medium', 'Small'],           # Ordinal
    'Target_Object': ['Cup', 'Bed', 'Sofa', 'Plate', 'Chair']                 # Target
}
df = pd.DataFrame(data)

df.head()

,Room_Type,Object_Color,Object_Size,Target_Object
0,Kitchen,Red,Small,Cup
1,Bedroom,Blue,Large,Bed
2,Living Room,Green,Medium,Sofa
3,Kitchen,Red,Medium,Plate
4,Bedroom,Blue,Small,Chair


In [15]:
# dataset that we can encode only these two features
df[['Room_Type', 'Object_Color']].head()


,Room_Type,Object_Color
0,Kitchen,Red
1,Bedroom,Blue
2,Living Room,Green
3,Kitchen,Red
4,Bedroom,Blue


In [13]:
df['Room_Type'].value_counts()


Room_Type
Kitchen        2
Bedroom        2
Living Room    1
Name: count, dtype: int64

In [14]:
df['Object_Color'].value_counts()


Object_Color
Red      2
Blue     2
Green    1
Name: count, dtype: int64

In [22]:
# Initialize OneHotEncoder
ohe = OneHotEncoder(sparse_output = False, drop = 'first')

# fit and transform nominal columns
nominal_features = df[['Room_Type', 'Object_Color']]
encoded_nominal = ohe.fit_transform(nominal_features)

# convert back to df for viewing
encoded_column = ohe.get_feature_names_out(['Room_Type', 'Object_Color'])
df_nominal_encoded = pd.DataFrame(encoded_nominal , columns = encoded_columns)
df_nominal_encoded

,Room_Type_Kitchen,Room_Type_Living Room,Object_Color_Green,Object_Color_Red
0,1.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0
2,0.0,1.0,1.0,0.0
3,1.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0


In [19]:
encoded_columns

array(['Room_Type_Kitchen', 'Room_Type_Living Room', 'Object_Color_Green',
       'Object_Color_Red'], dtype=object)

## Nominal Encoding:

In [37]:
from sklearn.preprocessing import OrdinalEncoder
df

,Room_Type,Object_Color,Object_Size,Target_Object,Object_Size_Encoded
0,Kitchen,Red,Small,Cup,0.0
1,Bedroom,Blue,Large,Bed,2.0
2,Living Room,Green,Medium,Sofa,1.0
3,Kitchen,Red,Medium,Plate,1.0
4,Bedroom,Blue,Small,Chair,0.0


In [38]:
df['Object_Size'].value_counts()

Object_Size
Small     2
Medium    2
Large     1
Name: count, dtype: int64

In [44]:
# I just encode one object-size column in nominal encoding because it has sequence or size 

# first define order
size_ord = [['Small', 'Medium', 'Large']]

# initialize encoder for specific categories
ord_encoder = OrdinalEncoder(categories = size_ord)

# Fit and Transform Ordinal column
df['Object_Size_Encoded'] = ord_encoder.fit_transform(df[['Object_Size']])
df[['Object_Size','Object_Size_Encoded']]

,Object_Size,Object_Size_Encoded
0,Small,0.0
1,Large,2.0
2,Medium,1.0
3,Medium,1.0
4,Small,0.0


## ColumnTransformer & Pipeline:
- In industry level development we don't use separatly ordinal or nomincal encoding.
- We use ColumnTransformer and Pipeline to make code error free and deployable

In [10]:
# importing import module for ColumnTransformer and pipeline
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier


In [4]:

# dataset generated all type of features that we required

data = {
    'Age': [25, 45, 35, 50, 23, 30, 40, 28, 55, 32],                    # Numerical
    'Income': [50000, 150000, 80000, 200000, 30000, 60000, 120000, 70000, 180000, 90000], # Numerical
    'City': ['Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Lahore', 'Islamabad', 'Karachi'], # Nominal
    'Education': ['Matric', 'Degree', 'Inter', 'Degree', 'Matric', 'Inter', 'Degree', 'Matric', 'Degree', 'Inter'], # Ordinal
    'Loan_Status': ['Denied', 'Approved', 'Approved', 'Approved', 'Denied', 'Denied', 'Approved', 'Denied', 'Approved', 'Approved'] # Target
}

df = pd.DataFrame(data)
df.head()

,Age,Income,City,Education,Loan_Status
0,25,50000,Karachi,Matric,Denied
1,45,150000,Lahore,Degree,Approved
2,35,80000,Islamabad,Inter,Approved
3,50,200000,Karachi,Degree,Approved
4,23,30000,Lahore,Matric,Denied


In [6]:
# separating features
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']


In [7]:
# LabelEncoder target ko 0 aur 1 mein map karta hai.
le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [8]:
# Train Test Spliting before encoding because we want to avoid data leakage and ensure that our model generalizes well to unseen data.

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# 5. Define Feature Groups

numeric_features = ['Age', 'Income']
nominal_features = ['City'] # Require OneHotEncoder
ordinal_features = ['Education'] # Require OrdinalEncoder

# Logic batani hoti hai ke chota degree konsa hai aur bara konsa
education_order = [['Matric', 'Inter', 'Degree']]


In [11]:
# 6. ColumnTransformer (The Heart of the Pipeline)
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), numeric_features),
        ('nom_encoder', OneHotEncoder(drop='first', handle_unknown='ignore'), nominal_features),
        ('ord_encoder', OrdinalEncoder(categories=education_order), ordinal_features)
    ])

In [12]:
# 7. Final Pipeline
# Preprocessor + ML Algorithm ko mila kar ek pipeline banayein

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [13]:
# 8. Train & Evaluate
# Sirf training data fit karein, pipeline khud sab ko encode/scale karegi!
model_pipeline.fit(X_train, y_train)

# Test Data par Accuracy Check karein
accuracy = model_pipeline.score(X_test, y_test)
print(f"\nModel Evaluation \nTest Accuracy: {accuracy * 100}%")



Model Evaluation 
Test Accuracy: 100.0%


In [14]:
# Dekhte hain ke model ko asal mein data kaisa gaya?

sample_data = X_train.head(1)
transformed_sample = preprocessor.transform(sample_data)

print(f"\nOriginal Row:\n{sample_data.to_string(index=False)}")
print(f"Transformed Numpy Array:\n{transformed_sample}")


Original Row:
 Age  Income      City Education
  30   60000 Islamabad     Inter
Transformed Numpy Array:
[[-0.34965069 -0.55629391  0.          0.          1.        ]]


In [15]:
import datetime
today = datetime.date.today()
print(f"Today's date: {today}")

Today's date: 2026-03-13
